# D-MTHD tweet benchmark on Google Colab

Use this if Kaggle verification is not possible. Colab's free tier gives a T4 GPU without identity verification.

1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Run the cells top to bottom (Runtime -> Run all). The first cell asks permission to use your Google Drive: results are written there, so a disconnected session loses nothing; just run all cells again and finished work is skipped.
3. Free Colab sessions can stop after a few hours. Re-running continues from the last finished run (and from the last finished epoch inside a run).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/dmthd-work'
import os; os.makedirs(ROOT, exist_ok=True); print('results will persist in', ROOT)

In [ ]:
import os, subprocess
if not os.path.exists('/content/dmthd-p3/src/dmthd'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mahdihasanshadi/THESIS.git', '/content/dmthd-p3'], check=True)
os.chdir('/content/dmthd-p3')
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'])
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime -> Change runtime type -> T4 GPU')

In [ ]:
import os, urllib.request
raw = f'{ROOT}/raw/cyberbullying_tweets.csv'
os.makedirs(os.path.dirname(raw), exist_ok=True)
if not os.path.exists(raw):
    urllib.request.urlretrieve('https://huggingface.co/datasets/mattematics/cyberbullying/resolve/main/cyberbullying_tweets.csv', raw)
print(os.path.getsize(raw), 'bytes')

In [ ]:
import os
os.environ['ROOT'] = ROOT
os.environ['PYTHONPATH'] = 'src'
os.environ['GPU'] = '1'
os.environ['SEEDS'] = '1,2,3'
os.environ['COMMITTEES'] = 'homo,hetero'
os.environ['MODES'] = 'ft,skd,uniform,dmthd'
!python kaggle/run_benchmark.py --dataset tweets --stage all --raw $ROOT/raw/cyberbullying_tweets.csv

In [ ]:
!python -m dmthd.aggregate --runs $ROOT/runs/tweets --out $ROOT/runs/tweets/summary.csv
print('Everything is in Google Drive under dmthd-work/runs/tweets; zip that folder and hand it over.')